# MoE Scaling Law — Training Notebook

Parameterised runner for the MoE scaling law experiment. Trains **one model per Kaggle session** (dual T4 constraint).

**Regimes:** Dense (D), MoE no balancing (M₀), MoE + aux loss (Mα), MoE + Loss-Free Balancing (Mℓ)

**Usage:** Place `run_config.json` in `/kaggle/working/` or the notebook directory, then run all cells.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Imports and setup
# ══════════════════════════════════════════════════════════════════════════════

import csv
import glob
import hashlib
import importlib
import json
import math
import os
import pathlib
import re
import shutil
import socket
import subprocess
import sys
import time
from datetime import datetime

import numpy as np
import torch
import yaml

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_GLOB_PATTERN    = "/kaggle/input/**/*.npy"
REPO_URL             = "https://github.com/iliasslasri/OLMoE.git"
OLMOE_DIR            = "/kaggle/working/OLMoE"
PIP_OVERRIDE_DIR     = "/kaggle/working/_pip_overrides"
TOKENIZER_URL        = "https://huggingface.co/allenai/gpt-neox-olmo-dolma-v1_5/resolve/main/tokenizer.json"
TOKENIZER_LOCAL_PATH = "tokenizers/allenai_gpt-neox-olmo-dolma-v1_5.json"
RESULTS_BASE         = "/kaggle/working/results"

OLMO_REPO_URL   = "https://github.com/Tristan22400/OLMo.git"
OLMO_BRANCH     = "routing/moe-strategies"
MEGABLOCKS_PIP  = "git+https://github.com/Tristan22400/megablocks.git@routing/auxiliary-loss-free"

# ── Config path ────────────────────────────────────────────────────────────────
CONFIG_JSON_PATH = os.environ.get("RUN_CONFIG", "/kaggle/working/run_config.json")
if not os.path.exists(CONFIG_JSON_PATH):
    CONFIG_JSON_PATH = "run_config.json"

print(f"Config path: {CONFIG_JSON_PATH}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
N_GPU = torch.cuda.device_count()
print(f"GPUs: {N_GPU}")
for i in range(N_GPU):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: Config loading — build run_name, create output folder
# ══════════════════════════════════════════════════════════════════════════════

with open(CONFIG_JSON_PATH) as f:
    run_cfg = json.load(f)

# Defaults
run_cfg.setdefault("batch_size", "auto")
run_cfg.setdefault("grad_accum_steps", 4)
run_cfg.setdefault("alpha_aux", 0.01)
run_cfg.setdefault("bias_update_rate", 0.001)
run_cfg.setdefault("max_steps", None)
run_cfg.setdefault("max_wall_clock_hours", 11.5)
run_cfg.setdefault("seq_len", 4096)
run_cfg.setdefault("step_in_protocol", None)

# ── Build descriptive run name ─────────────────────────────────────────────────
REGIME_SHORT = {
    "dense": "dense",
    "M0": "M0",
    "M_alpha": "Malpha",
    "M_lfb": "Mlfb",
}

regime = run_cfg["regime"]
C = run_cfg["compute_budget_flops"]
N_a = run_cfg["N_a"]
D = run_cfg["D"]
A = run_cfg["A"]
G = run_cfg["G"]
K = run_cfg["K"]
E = run_cfg["E"]
eta = run_cfg["learning_rate"]

run_name = (
    f"{REGIME_SHORT[regime]}_C{C:.0e}_Na{N_a:.0e}_D{D:.0e}"
    f"_A{A}_G{G}_K{K}_E{E}_lr{eta:.1e}"
)
# Clean up scientific notation (remove +0 etc)
run_name = run_name.replace("+0", "").replace("+", "")

OUTPUT_DIR = os.path.join(RESULTS_BASE, run_name)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save config echo
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(run_cfg, f, indent=2)

# Deterministic seed from run_name
SEED = int(hashlib.sha256(run_name.encode()).hexdigest(), 16) % (2**31)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Run name: {run_name}")
print(f"Output:   {OUTPUT_DIR}")
print(f"Seed:     {SEED}")
print(f"Regime:   {regime}")
print(f"Budget:   C={C:.2e}, N_a={N_a:.2e}, D={D:.2e}")
print(f"Arch:     A={A}, G={G}, K={K}, E={E}")
print(f"LR:       {eta}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: GRID GENERATION — RUN INDEPENDENTLY
# ══════════════════════════════════════════════════════════════════════════════
# This cell generates all run_config.json files for the full experiment.
# It does NOT depend on cells above. Run it once to produce all configs.
# Set GENERATE_GRID = True to activate.

GENERATE_GRID = False  # <-- Set to True to generate configs

if GENERATE_GRID:
    import csv
    import json
    import math
    import os
    from itertools import product

    CONFIGS_DIR = "./configs"
    os.makedirs(CONFIGS_DIR, exist_ok=True)

    grid_rows = []  # for experiment_grid.csv

    def eta_opt(C):
        """LR scaling law: eta = 1.1576 * C^(-0.1529)"""
        return 1.1576 * C ** (-0.1529)

    def solve_architecture(N_a_target, A, G, n_layers=16, seq_len=4096):
        """
        Solve for (d_model, d_expert, K, E) given target (N_a, A, G).
        
        Identities:
          G = 2 * d_model / d_expert
          A = K / E
          N_a = L * (4*d_model^2 + 2*K*d_model*d_expert)
              = L * d_model^2 * (4 + 2*K*2/G)  [since d_expert = 2*d_model/G]
              = L * d_model^2 * (4 + 4*K/G)
        """
        L = n_layers
        # For dense: K=1, E=1, A=1, G=1 => d_expert = 2*d_model
        # N_a = L * (4*d_model^2 + 2*1*d_model*2*d_model) = L * 8 * d_model^2
        if A >= 1.0 - 1e-6:  # Dense
            K_val = 1
            E_val = 1
            # N_a = L * d_model^2 * (4 + 4*K/G)
            # For dense G=1, K=1: N_a = L * d_model^2 * 8
            d_model = int(round(math.sqrt(N_a_target / (L * 8))))
            # Round to multiple of 64 for efficiency
            d_model = max(64, 64 * round(d_model / 64))
            d_expert = 2 * d_model  # G=1
            return d_model, d_expert, K_val, E_val
        else:
            # MoE case
            # Pick K first (typically 2), then E = K/A
            K_val = 2  # default
            E_val = int(round(K_val / A))
            # Ensure A = K/E is exact
            A_actual = K_val / E_val
            
            # d_expert = 2 * d_model / G
            # N_a = L * (4*d_model^2 + 2*K*d_model*(2*d_model/G))
            #     = L * d_model^2 * (4 + 4*K/G)
            factor = 4 + 4 * K_val / G
            d_model = int(round(math.sqrt(N_a_target / (L * factor))))
            d_model = max(64, 64 * round(d_model / 64))
            d_expert = max(64, int(round(2 * d_model / G)))
            # Recompute G to be exact
            return d_model, d_expert, K_val, E_val

    def compute_flops(d_model, d_expert, K, n_layers, D, T):
        """Exact FLOPs (Eq. 5)"""
        return 3 * D * n_layers * (8 * d_model**2 + 4 * T * d_model + 4 * K * d_model * d_expert)

    def make_config(regime, C_budget, N_a_target, D_tokens, A, G,
                    n_layers=16, seq_len=4096, step=None, **overrides):
        d_model, d_expert, K_val, E_val = solve_architecture(N_a_target, A, G, n_layers, seq_len)
        lr = eta_opt(C_budget)

        cfg = {
            "regime": regime,
            "compute_budget_flops": C_budget,
            "N_a": N_a_target,
            "D": D_tokens,
            "A": A,
            "G": G,
            "K": K_val,
            "E": E_val,
            "d_model": d_model,
            "d_expert": d_expert,
            "n_layers": n_layers,
            "seq_len": seq_len,
            "learning_rate": lr,
            "batch_size": "auto",
            "grad_accum_steps": 4,
            "alpha_aux": 0.01 if regime == "M_alpha" else 0.0,
            "bias_update_rate": 0.001 if regime == "M_lfb" else 0.0,
            "max_steps": None,
            "max_wall_clock_hours": 11.5,
            "step_in_protocol": step,
        }
        cfg.update(overrides)
        return cfg

    def save_config(cfg):
        regime_short = {"dense": "dense", "M0": "M0", "M_alpha": "Malpha", "M_lfb": "Mlfb"}
        name = (
            f"{regime_short[cfg['regime']]}_C{cfg['compute_budget_flops']:.0e}"
            f"_Na{cfg['N_a']:.0e}_D{cfg['D']:.0e}"
            f"_A{cfg['A']}_G{cfg['G']}_K{cfg['K']}_E{cfg['E']}"
            f"_lr{cfg['learning_rate']:.1e}"
        ).replace("+0", "").replace("+", "")

        path = os.path.join(CONFIGS_DIR, f"{name}.json")
        with open(path, "w") as f:
            json.dump(cfg, f, indent=2)

        grid_rows.append({
            "run_name": name, "regime": cfg["regime"],
            "C": cfg["compute_budget_flops"], "N_a": cfg["N_a"], "D": cfg["D"],
            "A": cfg["A"], "G": cfg["G"], "K": cfg["K"], "E": cfg["E"],
            "d_model": cfg["d_model"], "d_expert": cfg["d_expert"],
            "n_layers": cfg["n_layers"], "eta": cfg["learning_rate"],
            "step_in_protocol": cfg.get("step_in_protocol"),
            "status": "pending",
        })
        return name

    # ── Step 0: LR Verification (3 budgets x 3 LRs = 9 runs) ──────────────────
    print("=== Step 0: LR Verification ===")
    lr_budgets = [1e18, 10**19.5, 1e21]
    N_a_ref = 5e7  # reference model size
    for C_k in lr_budgets:
        eta_ref = eta_opt(C_k)
        for lr_mult in [0.5, 1.0, 2.0]:
            lr = eta_ref * lr_mult
            # Compute D from C budget
            d_m, d_e, K_v, E_v = solve_architecture(N_a_ref, 1/32, 4, 16, 4096)
            M = 2 * N_a_ref + 4 * 16 * 4096 * d_m
            D_tok = C_k / (3 * M)
            cfg = make_config("M_lfb", C_k, N_a_ref, D_tok, 1/32, 4, step=0,
                              learning_rate=lr)
            name = save_config(cfg)
            print(f"  {name}")

    # ── Step 1: Dense Baseline (4 budgets x 5 allocations = 20 runs) ───────────
    print("\n=== Step 1: Dense Baseline ===")
    dense_budgets = [1e18, 1e19, 1e20, 1e21]
    N_a_multipliers = [0.2, 0.5, 1.0, 2.0, 5.0]  # relative to Chinchilla-optimal

    for C_k in dense_budgets:
        # Chinchilla-optimal N_a: approximate as proportional to C^0.5
        N_a_chin = 0.05 * C_k**0.5  # rough estimate
        for mult in N_a_multipliers:
            N_a_target = N_a_chin * mult
            d_m, d_e, K_v, E_v = solve_architecture(N_a_target, 1.0, 1, 16, 4096)
            M = 2 * N_a_target + 4 * 16 * 4096 * d_m
            D_tok = C_k / (3 * M)
            cfg = make_config("dense", C_k, N_a_target, D_tok, 1.0, 1, step=1)
            name = save_config(cfg)
            print(f"  {name}")

    # ── Step 2: Activation Ratio Sweep ─────────────────────────────────────────
    print("\n=== Step 2: Activation Ratio Sweep ===")
    moe_regimes = ["M0", "M_alpha", "M_lfb"]
    A_values = [1/64, 1/32, 1/16, 1/8, 1/4, 1/2, 1.0]
    primary_budget = 1e20
    stability_budgets = [1e19, 1e21]

    # Use Chinchilla-optimal (N_a, D) from dense at primary budget
    N_a_primary = 0.05 * primary_budget**0.5
    d_m_p, _, _, _ = solve_architecture(N_a_primary, 1/32, 4, 16, 4096)
    M_p = 2 * N_a_primary + 4 * 16 * 4096 * d_m_p
    D_primary = primary_budget / (3 * M_p)

    for reg in moe_regimes:
        for A_val in A_values:
            cfg = make_config(reg, primary_budget, N_a_primary, D_primary, A_val, 4, step=2)
            name = save_config(cfg)
            print(f"  {name}")

        # Stability checks at other budgets
        for C_k in stability_budgets:
            N_a_k = 0.05 * C_k**0.5
            d_m_k, _, _, _ = solve_architecture(N_a_k, 1/32, 4, 16, 4096)
            M_k = 2 * N_a_k + 4 * 16 * 4096 * d_m_k
            D_k = C_k / (3 * M_k)
            for A_val in A_values:
                cfg = make_config(reg, C_k, N_a_k, D_k, A_val, 4, step=2)
                name = save_config(cfg)

    # ── Step 3: Granularity Sweep ──────────────────────────────────────────────
    print("\n=== Step 3: Granularity Sweep ===")
    G_values = [1, 2, 4, 8, 12, 16]
    A_fixed = 1/32

    for reg in moe_regimes:
        for G_val in G_values:
            cfg = make_config(reg, primary_budget, N_a_primary, D_primary,
                              A_fixed, G_val, step=3)
            name = save_config(cfg)
            print(f"  {name}")

    # ── Write master CSV ───────────────────────────────────────────────────────
    csv_path = "experiment_grid.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=grid_rows[0].keys())
        writer.writeheader()
        writer.writerows(grid_rows)

    print(f"\nGenerated {len(grid_rows)} configs in {CONFIGS_DIR}/")
    print(f"Master CSV: {csv_path}")

else:
    print("Grid generation skipped (set GENERATE_GRID = True to run)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: Environment setup (clone, install, tokenizer)
# === MODIFIED FROM BASELINE ===
# Identical to baseline cells 7-14 but parameterised by run_cfg.
# ══════════════════════════════════════════════════════════════════════════════

TOKENIZED_DATA_PATHS = sorted(glob.glob(DATA_GLOB_PATTERN))
assert len(TOKENIZED_DATA_PATHS) > 0, f"No .npy files at {DATA_GLOB_PATTERN}"
print(f"Found {len(TOKENIZED_DATA_PATHS)} data shard(s)")

# W&B (optional)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
except Exception:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B disabled (no API key)")

os.environ.setdefault("WANDB_PROJECT", "moe-scaling-laws")
os.environ.setdefault("WANDB_ENTITY", "iliass-lasri-team")

# Clone
OLMO_SRC = f"{OLMOE_DIR}/OLMo"
if not os.path.exists(OLMOE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, OLMOE_DIR], check=True)
    subprocess.run(["git", "config", "submodule.OLMo.url", OLMO_REPO_URL],
                   cwd=OLMOE_DIR, check=True)
    subprocess.run(["git", "submodule", "update", "--init"], cwd=OLMOE_DIR, check=True)
    subprocess.run(["git", "fetch", "origin", OLMO_BRANCH],
                   cwd=OLMO_SRC, check=True)
    subprocess.run(["git", "checkout", OLMO_BRANCH],
                   cwd=OLMO_SRC, check=True)
else:
    print("Repo already present")

os.chdir(OLMOE_DIR)

# Dependencies (from baseline cells 10-13)
os.makedirs(PIP_OVERRIDE_DIR, exist_ok=True)

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"pip failed: {' '.join(args)}")

def pip_install_isolated(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--target", PIP_OVERRIDE_DIR, "--no-deps"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"pip_install_isolated failed")

def purge_cached_modules(*prefixes):
    stale = [k for k in sys.modules if any(k == p or k.startswith(p + ".") for p in prefixes)]
    for k in stale:
        del sys.modules[k]

print("[1/5] Installing OLMo[train]...")
pip_install("-e", "OLMo[train]")
if OLMO_SRC not in sys.path:
    sys.path.insert(1, OLMO_SRC)

print("[2/5] Installing megablocks...")
pip_install("--force-reinstall", MEGABLOCKS_PIP)

print("[3/5] Detecting torch version...")
detect = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                        capture_output=True, text=True, check=True)
installed_torch = detect.stdout.strip()
cuda_tag = installed_torch.split("+")[1] if "+" in installed_torch else "cpu"

print("[4/5] Reinstalling torchvision...")
pip_install("--force-reinstall", "torchvision",
            "--index-url", f"https://download.pytorch.org/whl/{cuda_tag}")
purge_cached_modules("torchvision")

print("[5/5] Fixing huggingface_hub...")
pip_install_isolated("huggingface_hub>=1.3.0,<2.0")
pip_install_isolated("tokenizers>=0.22.0,<=0.23.0")
if PIP_OVERRIDE_DIR not in sys.path:
    sys.path.insert(0, PIP_OVERRIDE_DIR)
purge_cached_modules("huggingface_hub", "tokenizers")

# Tokenizer
os.makedirs(os.path.dirname(TOKENIZER_LOCAL_PATH), exist_ok=True)
if not os.path.exists(TOKENIZER_LOCAL_PATH):
    subprocess.run(["wget", "-q", TOKENIZER_URL, "-O", TOKENIZER_LOCAL_PATH], check=True)

print("Environment ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: Model builder — config → OLMoE model, identity verification
# === MODIFIED FROM BASELINE ===
# ══════════════════════════════════════════════════════════════════════════════

def compute_N_a(L, d_model, d_expert, K):
    """Active params: N_a = L * (4*d_model^2 + 2*K*d_model*d_expert)"""
    return L * (4 * d_model**2 + 2 * K * d_model * d_expert)

def compute_flops_exact(D, L, d_model, d_expert, K, T):
    """Exact FLOPs: C = 3*D*L*(8*d_model^2 + 4*T*d_model + 4*K*d_model*d_expert)"""
    return 3 * D * L * (8 * d_model**2 + 4 * T * d_model + 4 * K * d_model * d_expert)

# Extract architecture params
d_model  = run_cfg["d_model"]
d_expert = run_cfg["d_expert"]
n_layers = run_cfg["n_layers"]
K_val    = run_cfg["K"]
E_val    = run_cfg["E"]
seq_len  = run_cfg["seq_len"]

# ── Structural identity verification (Sec F) ──────────────────────────────────
N_a_computed = compute_N_a(n_layers, d_model, d_expert, K_val)
A_computed = K_val / E_val
G_computed = 2 * d_model / d_expert
C_estimated = compute_flops_exact(D, n_layers, d_model, d_expert, K_val, seq_len)

checks = []

# N_a check (within 1%)
na_err = abs(N_a_computed - N_a) / N_a
checks.append(("N_a", na_err < 0.01, f"computed={N_a_computed:.0f}, config={N_a:.0f}, err={na_err:.4f}"))

# A check
a_err = abs(A_computed - A) / max(A, 1e-10)
checks.append(("A=K/E", a_err < 0.01, f"computed={A_computed:.4f}, config={A:.4f}"))

# G check
g_err = abs(G_computed - G) / max(G, 1e-10)
checks.append(("G=2d/d_e", g_err < 0.01, f"computed={G_computed:.4f}, config={G:.4f}"))

# FLOPs check (within 5%)
c_err = abs(C_estimated - C) / C
checks.append(("FLOPs", c_err < 0.05, f"estimated={C_estimated:.2e}, config={C:.2e}, err={c_err:.4f}"))

print("=== Structural Identity Verification ===")
all_pass = True
for name, passed, msg in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  {name}: {status} — {msg}")
    if not passed:
        all_pass = False

if not all_pass:
    raise RuntimeError("Structural identity checks FAILED. Aborting.")

# Write metadata
git_hash = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
metadata = (
    f"run_name: {run_name}\n"
    f"timestamp: {datetime.now().isoformat()}\n"
    f"git_hash: {git_hash}\n"
    f"gpu_info: {gpu_info}\n"
    f"seed: {SEED}\n"
    f"N_a_computed: {N_a_computed}\n"
    f"N_a_config: {N_a}\n"
    f"A_computed: {A_computed}\n"
    f"G_computed: {G_computed}\n"
    f"C_estimated: {C_estimated:.4e}\n"
    f"all_checks_pass: {all_pass}\n"
)
with open(os.path.join(OUTPUT_DIR, "metadata.txt"), "w") as f:
    f.write(metadata)

# Parameter counts
N_attn = 4 * n_layers * d_model**2
N_expert_single = 2 * d_model * d_expert
N_total = N_attn + E_val * N_expert_single
print(f"\nN_a = {N_a_computed:,.0f}, N_total = {N_total:,.0f}")
print(f"N_attn = {N_attn:,.0f}, N_expert (each) = {N_expert_single:,.0f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: Regime-specific YAML config generation
# === MODIFIED FROM BASELINE ===
# Configures the OLMoE YAML differently per regime.
# ══════════════════════════════════════════════════════════════════════════════

CONFIG_PATH = pathlib.Path(OLMOE_DIR) / "configs/olmoe-small.yml"
cfg = yaml.safe_load(CONFIG_PATH.read_text())

# ── Common settings ───────────────────────────────────────────────────────────
cfg["data"]["paths"]                 = list(TOKENIZED_DATA_PATHS)
cfg["model"]["d_model"]             = d_model
cfg["model"]["n_layers"]            = n_layers
cfg["model"]["max_sequence_length"] = seq_len
cfg["model"]["mlp_ratio"]           = 1  # d_expert set via moe config
cfg["model"]["moe_mlp_impl"]        = "sparse"
cfg["model"]["moe_dropless"]        = False  # T4 doesn't support sparse kernels well
cfg["precision"]                    = "amp_fp16"
cfg["activation_checkpointing"]     = "fine_grained"
cfg["compile"]                      = None
cfg["seed"]                         = SEED
cfg["save_folder"]                  = OUTPUT_DIR
cfg["run_name"]                     = run_name

# LR and scheduler
cfg["optimizer"]["learning_rate"]    = eta

# Compute max steps from D and batch size
# Batch size
if run_cfg["batch_size"] == "auto":
    # T4: 16GB. Conservative: microbatch = 2 for small models, 1 for large
    if d_model <= 1024:
        micro_bs = 4
    elif d_model <= 2048:
        micro_bs = 2
    else:
        micro_bs = 1
else:
    micro_bs = int(run_cfg["batch_size"])

grad_accum = run_cfg["grad_accum_steps"]
device_bs = micro_bs * grad_accum
global_bs = device_bs * max(N_GPU, 1)
tokens_per_step = global_bs * seq_len

if run_cfg["max_steps"] is not None:
    max_steps = run_cfg["max_steps"]
else:
    max_steps = int(math.ceil(D / tokens_per_step))

cfg["global_train_batch_size"]       = global_bs
cfg["device_train_microbatch_size"]  = micro_bs
cfg["max_duration"]                  = max_steps
cfg["save_interval"]                 = max(max_steps // 10, 1000)

# Scheduler: WSD (warmup-stable-decay)
warmup_steps = min(2500, max_steps // 10)
decay_start = int(max_steps * 0.98)
cfg["scheduler"] = {
    "name": "cosine_with_warmup",
    "units": "steps",
    "t_warmup": warmup_steps,
    "t_max": max_steps,
    "alpha_f": 0.1,
}

# Eval
cfg["eval_interval"] = max(max_steps // 20, 500)
cfg["eval_subset_num_batches"] = 50

# Remove fsdp.use_orig_params if present
cfg.get("fsdp", {}).pop("use_orig_params", None)

# ── Regime-specific ───────────────────────────────────────────────────────────
if regime == "dense":
    # Dense: 1 expert, no MoE losses
    cfg["model"]["moe_num_experts"]    = 1
    cfg["model"]["moe_top_k"]          = 1
    cfg["model"]["moe_loss_weight"]    = 0.0
    cfg["model"]["moe_zloss_weight"]   = None
    cfg["model"]["moe_routing_type"]   = "learned"
    cfg["model"]["block_type"]         = "moe"  # OLMoE always uses moe block
    print("Regime: DENSE (E=1, K=1, no aux losses)")

elif regime == "M0":
    # MoE, no balancing
    cfg["model"]["moe_num_experts"]    = E_val
    cfg["model"]["moe_top_k"]          = K_val
    cfg["model"]["moe_loss_weight"]    = 0.0  # no LB loss
    cfg["model"]["moe_zloss_weight"]   = 0.001  # keep z-loss
    cfg["model"]["moe_routing_type"]   = "learned"
    print(f"Regime: M0 (E={E_val}, K={K_val}, no LB loss, z-loss=0.001)")

elif regime == "M_alpha":
    # MoE + auxiliary loss
    alpha_aux = run_cfg["alpha_aux"]
    cfg["model"]["moe_num_experts"]    = E_val
    cfg["model"]["moe_top_k"]          = K_val
    cfg["model"]["moe_loss_weight"]    = alpha_aux
    cfg["model"]["moe_zloss_weight"]   = 0.001
    cfg["model"]["moe_routing_type"]   = "learned"
    print(f"Regime: M_alpha (E={E_val}, K={K_val}, alpha_aux={alpha_aux}, z-loss=0.001)")

elif regime == "M_lfb":
    # MoE + Loss-Free Balancing
    bias_rate = run_cfg["bias_update_rate"]
    cfg["model"]["moe_num_experts"]       = E_val
    cfg["model"]["moe_top_k"]             = K_val
    cfg["model"]["moe_loss_weight"]       = 0.0  # LB loss computed but NOT added
    cfg["model"]["moe_zloss_weight"]      = 0.001  # z-loss still active
    cfg["model"]["moe_routing_type"]      = "loss_free"
    cfg["model"]["moe_bias_update_speed"] = bias_rate
    print(f"Regime: M_lfb (E={E_val}, K={K_val}, LFB, bias_rate={bias_rate}, z-loss=0.001)")

else:
    raise ValueError(f"Unknown regime: {regime}")

# Write YAML
config_yaml_text = yaml.dump(cfg, default_flow_style=False, sort_keys=False)
config_yaml_text = re.sub(r"'(\$\{[^}]+\}[^']*)',", r"\1", config_yaml_text)
CONFIG_PATH.write_text(config_yaml_text)

print(f"\nConfig written to {CONFIG_PATH}")
print(f"Max steps: {max_steps}, tokens/step: {tokens_per_step:,}")
print(f"Global BS: {global_bs}, Micro BS: {micro_bs}, Grad accum: {grad_accum}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: Dry-run validation (from baseline)
# ══════════════════════════════════════════════════════════════════════════════

dry_run_env = os.environ.copy()
dry_run_env["OLMO_TASK"]       = "model"
dry_run_env["PYTHONPATH"]      = f"{PIP_OVERRIDE_DIR}:{OLMO_SRC}:" + dry_run_env.get("PYTHONPATH", "")
dry_run_env["OMP_NUM_THREADS"] = "1"

result = subprocess.run(
    [sys.executable, f"{OLMOE_DIR}/OLMo/scripts/train.py", str(CONFIG_PATH)],
    capture_output=True, text=True, env=dry_run_env, cwd=OLMOE_DIR,
)

IMPORT_ERROR_MARKERS = ["ModuleNotFoundError", "ImportError", "cannot import name"]
CONFIG_ERROR_MARKERS = ["OmegaConf", "yaml", "KeyError", "MissingMandatoryValue"]

has_import_error = any(m in result.stderr for m in IMPORT_ERROR_MARKERS)
has_config_error = any(m in result.stderr for m in CONFIG_ERROR_MARKERS)
reached_distributed = "RANK expected, but not set" in result.stderr

if has_import_error:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("Import error in dry run")
if has_config_error:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("Config error in dry run")
if reached_distributed:
    print("Dry run PASSED: all imports and config loaded OK.")
elif result.returncode == 0:
    print("Dry run PASSED cleanly.")
else:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError(f"Unexpected failure (exit {result.returncode})")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Training loop (wall-clock-aware)
# === MODIFIED FROM BASELINE ===
# Adds wall-clock monitoring and structured logging.
# ══════════════════════════════════════════════════════════════════════════════

def find_free_port():
    with socket.socket() as s:
        s.bind(('', 0))
        return s.getsockname()[1]

rdzv_port = find_free_port()
max_wall_seconds = run_cfg["max_wall_clock_hours"] * 3600
start_time = time.time()

print(f"Launching training on {N_GPU} GPU(s)...")
print(f"Wall-clock limit: {run_cfg['max_wall_clock_hours']}h ({max_wall_seconds:.0f}s)")
print(f"rdzv port: {rdzv_port}")

train_env = os.environ.copy()
train_env["OLMO_TASK"]               = "model"
train_env["PYTHONPATH"]              = f"{PIP_OVERRIDE_DIR}:{OLMO_SRC}:" + train_env.get("PYTHONPATH", "")
train_env["OMP_NUM_THREADS"]         = "4"
train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set time limit in OLMo config (it has native support)
cfg_reload = yaml.safe_load(CONFIG_PATH.read_text())
cfg_reload["time_limit"] = int(max_wall_seconds - 300)  # 5min safety margin
CONFIG_PATH.write_text(yaml.dump(cfg_reload, default_flow_style=False, sort_keys=False))

train_cmd = [
    sys.executable, "-m", "torch.distributed.run",
    f"--nproc-per-node={max(N_GPU, 1)}",
    "--nnodes=1", "--node_rank=0",
    "--rdzv_backend=c10d",
    f"--rdzv_endpoint=localhost:{rdzv_port}",
    "OLMo/scripts/train.py",
    str(CONFIG_PATH),
]

print("Command:", " ".join(train_cmd))
print("-" * 60)

# Logging files
training_log_path = os.path.join(OUTPUT_DIR, "training_log.csv")
with open(training_log_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step", "train_loss", "val_loss", "lr", "grad_norm", "wall_time"])

process = subprocess.Popen(
    train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=train_env,
)

completed = True
early_stop_reason = None
last_step = 0
last_train_loss = None
last_val_loss = None

for line in process.stdout:
    print(line, end="", flush=True)

    # Parse training metrics from OLMo output
    if "train/loss" in line:
        try:
            # OLMo logs: [step=NNN] train/loss=X.XX ...
            step_match = re.search(r'\[step=(\d+)\]', line)
            loss_match = re.search(r'train/loss[= ]+(\d+\.\d+)', line)
            if step_match and loss_match:
                last_step = int(step_match.group(1))
                last_train_loss = float(loss_match.group(1))
        except (ValueError, AttributeError):
            pass

    if "eval/loss" in line or "val/loss" in line:
        try:
            loss_match = re.search(r'(?:eval|val)/loss[= ]+(\d+\.\d+)', line)
            if loss_match:
                last_val_loss = float(loss_match.group(1))
        except (ValueError, AttributeError):
            pass

    # Wall-clock check
    elapsed = time.time() - start_time
    if elapsed > max_wall_seconds:
        print(f"\nWALL CLOCK LIMIT ({run_cfg['max_wall_clock_hours']}h) reached. Stopping...")
        process.terminate()
        completed = False
        early_stop_reason = "wall_clock_limit"
        break

process.wait()
wall_seconds = time.time() - start_time

if process.returncode != 0 and completed:
    # Check if it was a graceful time-limit exit
    if process.returncode in (143, -15):  # SIGTERM
        completed = False
        early_stop_reason = "wall_clock_limit"
    else:
        print(f"WARNING: Training exited with code {process.returncode}")

print(f"\nTraining {'completed' if completed else 'stopped early'}.")
print(f"Wall time: {wall_seconds/3600:.2f}h")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Evaluation and output saving
# ══════════════════════════════════════════════════════════════════════════════

# Compute actual values
actual_tokens = last_step * tokens_per_step if last_step > 0 else 0
actual_flops = compute_flops_exact(actual_tokens, n_layers, d_model, d_expert, K_val, seq_len)
flops_per_step = compute_flops_exact(tokens_per_step, n_layers, d_model, d_expert, K_val, seq_len)

# Build results JSON
results = {
    "run_name": run_name,
    "config": run_cfg,

    "val_loss_final": last_val_loss,
    "train_loss_final": last_train_loss,
    "val_loss_curve": [],  # populated from training_log.csv if available

    "actual_flops": actual_flops,
    "actual_tokens_seen": actual_tokens,
    "flops_per_step": flops_per_step,

    "actual_N_a": N_a_computed,
    "actual_N_total": N_total,
    "actual_A": A_computed,
    "actual_G": G_computed,
    "identity_checks_passed": all_pass,

    # MoE-specific (null for dense)
    "max_violation_global": None,
    "router_z_loss_final": None,
    "expert_bias_final": None,

    "num_steps": last_step,
    "effective_batch_size_tokens": tokens_per_step,
    "wall_clock_seconds": wall_seconds,
    "completed": completed,
    "early_stopped_reason": early_stop_reason,
}

# Try to read WandB or checkpoint for MoE stats
if regime != "dense":
    # Look for expert load logs from OLMo's saved outputs
    expert_log_files = glob.glob(os.path.join(OUTPUT_DIR, "**/expert_*.csv"), recursive=True)
    if expert_log_files:
        results["expert_load_log_available"] = True

# Save results JSON
results_path = os.path.join(OUTPUT_DIR, "results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {results_path}")
print(f"\n=== Summary ===")
print(f"  Run:         {run_name}")
print(f"  Regime:      {regime}")
print(f"  Steps:       {last_step}")
print(f"  Train loss:  {last_train_loss}")
print(f"  Val loss:    {last_val_loss}")
print(f"  Tokens:      {actual_tokens:,.0f}")
print(f"  FLOPs:       {actual_flops:.2e}")
print(f"  Wall time:   {wall_seconds/3600:.2f}h")
print(f"  Completed:   {completed}")

# List output files
print(f"\n=== Output files ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f}  ({size/1024:.1f} KB)")